# RL Beat Generation — PPO Training · Phase 2 (Colab)

Trains a PPO agent to generate beats on an **8×16 grid** (drums + Bass, Melody, Pad, FX)
using a hybrid reward: rule-based musical structure (drums + melodic elements) +
a pre-trained Phase 2 transformer discriminator.

> **Runtime:** `Runtime → Change runtime type → A100 GPU` (T4 may OOM at L=8 with 32 episodes/epoch — reduce to 16 if needed)  
> **Prerequisites:** Run `train_discriminator_colab.ipynb` first to produce `discriminator_phase2.pt`.

## 1 · Setup

In [ ]:
!git clone -b feature/phase2-updates https://github.com/Atharv-Girish-Chaudhary/rl-beat-generation.git
%cd rl-beat-generation
!pip install -e . -q

## 2 · Mount Drive + Copy Assets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, zipfile
DRIVE_PATH = "/content/drive/MyDrive/rl_term_project"

# ── Audio samples ──────────────────────────────────────────────────────────
zip_path = f"{DRIVE_PATH}/data/samples.zip"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/rl-beat-generation")
    print("✅ Samples extracted")
else:
    print("⚠️  samples.zip not found — audio generation will be skipped")

# ── Discriminator checkpoint ───────────────────────────────────────────────
os.makedirs("outputs/checkpoints", exist_ok=True)
disc_src = f"{DRIVE_PATH}/checkpoints/discriminator_phase2.pt"
disc_dst = "outputs/checkpoints/discriminator_phase2.pt"
if os.path.exists(disc_src):
    shutil.copy(disc_src, disc_dst)
    print(f"✅ Discriminator checkpoint copied")
else:
    print("❌ discriminator_phase2.pt not found in Drive — run train_discriminator_colab.ipynb first!")

## 3 · Resume from Checkpoint (optional)

If a previous Colab session saved actor/critic checkpoints to Drive, this cell
restores them so training continues from where it left off rather than restarting.
Safe to run even on a fresh session — it will no-op if no checkpoints are found.

In [ ]:
import os, shutil

RESUME = False  # set True to restore actor/critic from Drive

ckpt_files = ["actor_best.pt", "critic_best.pt"]
if RESUME:
    for f in ckpt_files:
        src = f"{DRIVE_PATH}/checkpoints/{f}"
        dst = f"outputs/checkpoints/{f}"
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"✅ Restored {f}")
        else:
            print(f"⚠️  {f} not found in Drive — will train from scratch")
else:
    print("ℹ️  RESUME=False — starting from scratch")

## 4 · Imports + Device Check

In [ ]:
import sys
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Device ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = "cuda"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("GPU  : Apple MPS")
else:
    device = "cpu"
    print("GPU  : none — running on CPU (training will be slow)")

# ── System RAM ────────────────────────────────────────────────────────────
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM  : {vm.total/1e9:.1f} GB total, {vm.available/1e9:.1f} GB available")
except ImportError:
    with open('/proc/meminfo') as f:
        lines = {k.strip(): v.strip() for k, v in
                 (l.split(':', 1) for l in f if ':' in l)}
    total_kb = int(lines.get('MemTotal', '0 kB').split()[0])
    avail_kb = int(lines.get('MemAvailable', '0 kB').split()[0])
    print(f"RAM  : {total_kb/1e6:.1f} GB total, {avail_kb/1e6:.1f} GB available")

print(f"\nPython : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

## 5 · Config

All hyperparameters in one place. `PHASE = 2` expands the grid to L=8 and
activates `_evaluate_melodic_elements()` inside `compute_reward()`.  
`ALPHA=0.5 / BETA=0.5` gives equal weight to rule-based and discriminator rewards.

> If you hit OOM on T4, set `EPISODES_PER_EPOCH = 16`.

In [ ]:
PHASE               = 2
EPOCHS              = 500
ALPHA               = 0.5   # rule-based reward weight
BETA                = 0.5   # discriminator reward weight
EPISODES_PER_EPOCH  = 32
LR_ACTOR            = 3e-4
LR_CRITIC           = 1e-3
DISC_PATH           = "outputs/checkpoints/discriminator_phase2.pt"

# Derived — never hardcode L below this cell
L = 4 if PHASE == 1 else 8
T = 16

print("Config:")
print(f"  phase={PHASE}  (L={L}, T={T})")
print(f"  alpha={ALPHA}, beta={BETA}  (rule vs discriminator weight)")
print(f"  epochs={EPOCHS}, episodes/epoch={EPISODES_PER_EPOCH}")
print(f"  lr_actor={LR_ACTOR}, lr_critic={LR_CRITIC}")
print(f"  disc_path={DISC_PATH}")

## 6 · Discriminator Load + Sanity Check

Verifies the Phase 2 discriminator scores a structured full-band beat
higher than silence before it is passed to the PPO reward function.

In [ ]:
from beat_rl.models import BeatDiscriminator

disc = BeatDiscriminator(
    num_instruments=L, num_steps=T,
    d_model=64, num_heads=4, num_blocks=2, d_ff=128
).to(device)
disc.load_state_dict(torch.load(DISC_PATH, map_location=device, weights_only=True))
disc.eval()
print(f"Loaded discriminator from {DISC_PATH}")

# ── Forward pass 1: completely silent grid ─────────────────────────────────
silent = torch.zeros(1, L, T, device=device)
with torch.no_grad():
    logit, _ = disc(silent)
score_silent = torch.sigmoid(logit).item()

# ── Forward pass 2: structured Phase 2 beat (drums + bass lock) ───────────
structured = torch.zeros(1, L, T)
structured[0, 0, [0, 4, 8, 12]] = 1.0          # kick   — quarter notes
structured[0, 1, [4, 12]]       = 1.0          # snare  — 2 & 4
structured[0, 2, list(range(0, 16, 2))] = 1.0  # hihat  — 8th notes
structured[0, 3, [4, 12]]       = 1.0          # clap   — with snare
if L == 8:
    structured[0, 4, [0, 4, 8, 12]] = 1.0      # bass   — locked to kick
with torch.no_grad():
    logit2, _ = disc(structured.to(device))
score_structured = torch.sigmoid(logit2).item()

print(f"\nOutput shape      : {logit.shape}  (batch=1, logit=1)")
print(f"Silent grid score : {score_silent:.4f}  (expect < 0.5)")
print(f"Structured score  : {score_structured:.4f}  (expect > silent)")

assert score_structured > score_silent, \
    "Discriminator sanity check failed: structured beat should score higher than silence"
print("\n✅ Sanity check passed — ready to train.")

## 7 · Training

Epoch logs stream as they arrive. Each line shows:
`Epoch N | Mean Reward | Actor Loss | Critic Loss`

A checkpoint is saved to Drive whenever mean reward improves.

Estimated wall-clock: ~4–5 h on T4, ~1.5 h on A100 (Phase 2 at L=8).

In [ ]:
import sys
sys.path.insert(0, '.')

from scripts.train_ppo import train_ppo

train_ppo(
    epochs=EPOCHS,
    episodes_per_epoch=EPISODES_PER_EPOCH,
    alpha=ALPHA,
    beta=BETA,
    pi_lr=LR_ACTOR,
    v_lr=LR_CRITIC,
    device=device,
    phase=PHASE,
)

## 8 · Save Checkpoints to Drive

Copies the best actor and critic weights back to Drive for persistence across sessions.

In [ ]:
import shutil, os

dest_dir = f"{DRIVE_PATH}/checkpoints"
os.makedirs(dest_dir, exist_ok=True)

for fname in ["actor_best.pt", "critic_best.pt"]:
    src = f"outputs/checkpoints/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{dest_dir}/{fname}")
        print(f"✅ Saved {fname} → Drive")
    else:
        print(f"⚠️  {fname} not found — training may not have completed a save yet")